In [ ]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import logging
from matplotlib import pylab
import os
import sys
import yaml


In [ ]:
sc.set_figure_params(dpi=100, facecolor='white', dpi_save=500)
pylab.rcParams['figure.figsize'] = (4, 4)
homeDir = os.getenv("HOME")

sys.path.insert(1, homeDir+"/utils/")


from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *

import rapids_singlecell as rsc


In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)
DS = "B19-25653_8um"
base_path = "/data/Spatial_Tx" 


import cupyx.scipy.sparse
import random
from scipy import sparse
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import cupy as cp

cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
import squidpy as sq

In [ ]:
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
anndata2ri.activate()
%load_ext rpy2.ipython
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)
DS = "B19-25653_8um"
base_path = "/data/Spatial_Tx" 


import cupyx.scipy.sparse
import random
from scipy import sparse
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import cupy as cp

cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
os.makedirs(f"./Deconvolution_{DS}/", exist_ok=True)


# Ref handling

In [ ]:
Ref = sc.read_h5ad(homeDir+f"/Biermann_Atlas/adatas/AVGcounts_bycelltypecurated.h5ad")


In [ ]:



import numpy as np
import pandas as pd
from scipy.sparse import issparse

# --- load pieces ---
X = Ref.layers["random_summedCounts"]          # shape: cells × genes
genes = Ref.var_names.astype(str)              # length G
barcodes = Ref.obs_names.astype(str)           # length N
cell_types = Ref.obs["celltypecurated"].astype(str).values

# --- per-cell UMI (sum over genes) ---
if issparse(X):
    # X is N×G; sum over axis=1 -> per cell
    nUMI = np.asarray(X.sum(axis=1)).ravel()
    # counts CSV should be genes×cells -> transpose
    counts_df = pd.DataFrame(X.T.toarray(), index=genes, columns=barcodes)
else:
    X = np.asarray(X)
    nUMI = X.sum(axis=1).astype(np.float64)
    counts_df = pd.DataFrame(X.T, index=genes, columns=barcodes)

# ensure integer counts (optional if already ints)
if not np.issubdtype(counts_df.values.dtype, np.integer):
    counts_df = counts_df.round().astype(np.int64)
    nUMI = counts_df.sum(axis=0).to_numpy()  # recompute from rounded counts

# write CSVs for spacexr
counts_out = counts_df.copy()
counts_out.insert(0, "gene", counts_out.index)
counts_out.to_csv(f"./Deconvolution_{DS}/reference_dge.csv", index=False)

meta = pd.DataFrame({
    "barcode": barcodes,
    "cluster": cell_types,
    "nUMI":    nUMI.astype(np.int64)
})
meta.to_csv(f"./Deconvolution_{DS}/reference_meta_data.csv", index=False)

print("Wrote dge.csv (genes×cells) and meta_data.csv")





# Spatial data handling

In [ ]:
base_path = "/data/Spatial_Tx" 

adata = sc.read_h5ad(homeDir+f"/{DS}_CleanAdata.h5ad")


import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import issparse



# ---- pull pieces ----
X = adata.layers["counts"]                 # counts matrix (spots × genes)
genes = adata.var_names.astype(str)        # length G
barcodes = adata.obs_names.astype(str)     # length N

# coords in adata.obsm["FULLRESspatial"] with 2 columns (x, y)
coords_arr = adata.obsm["spatial"]
coords = pd.DataFrame(coords_arr, index=barcodes, columns=["xcoord", "ycoord"])

# ---- per-spot nUMI (sum across genes) ----

sparse.save_npz(f"./Deconvolution_{DS}/{DS}_spots_counts.npz", X)
coords.to_csv(f"./Deconvolution_{DS}/{DS}_spots_coords.tsv")
pd.Series(genes).to_csv(f"./Deconvolution_{DS}/{DS}_spots_var.tsv")
pd.Series(barcodes).to_csv(f"./Deconvolution_{DS}/{DS}_spots_obs.tsv")


